In [ ]:
# environment setup using python virtual environments and pip package manager
%pip install pandas seaborn matplotlib numpy Ipython


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Assignment 2  - CSI4142
Student names: Sameed Jafri (300253861), Jaime Bly (300231604)

work spilt:
- dataset 1: Sameed Jafri
- dataset 2: Jamie Bly



### Dataset 1
Description: New York City Airbnb Open Data

Dataset Name: New York City Airbnb Open Data (2019) 


Author: Compiled by Kaggle user Dgomonov, utilizing public data originally scraped from Airbnb.


Purpose: This is a real-world, non-synthetic dataset created to explore, visualize, and analyze Airbnb listing activity, pricing metrics, and geographic trends across New York City.


Shape: The dataset contains exactly 48,895 rows and 16 columns.


Features and Descriptions 

- id: Unique identifier for the Airbnb listing (Numerical)

- name: The title or name of the listing (Categorical / Text)

- host_id: Unique identifier for the host (Numerical)

- host_name: First name of the host (Categorical / Text)

- neighbourhood_group: The NYC borough where the listing is located (e.g., Manhattan, Brooklyn) (Categorical)

- neighbourhood: The specific neighborhood (e.g., Harlem, Williamsburg) (Categorical)

- latitude: Latitude coordinate of the listing (Numerical)

- longitude: Longitude coordinate of the listing (Numerical)

- room_type: The type of space offered (e.g., Entire home/apt, Private room) (Categorical)

- price: Nightly price of the listing in USD (Numerical)

- minimum_nights: The minimum number of nights required to book a stay (Numerical)

- number_of_reviews: Total number of reviews the listing has received (Numerical)

- last_review: The date of the most recent review formatted as YYYY-MM-DD (Categorical / Date)

- reviews_per_month: The average number of reviews the listing receives per month (Numerical)

- calculated_host_listings_count: The total amount of listings the host operates in NYC (Numerical)

- availability_365: The number of days the listing is available for booking throughout the year (Numerical)


In [6]:
# loading the dataset 1 and creating a copy of it for dirty dataset
import pandas as pd
import numpy as np
from IPython.display import display

# loading the dataset
file_path = "dataset1/AB_NYC_2019.csv"

df_clean = pd.read_csv(file_path)

df_dirty = df_clean.copy()
print(f"Dataset loaded successfully with {df_clean.shape[0]} rows and {df_clean.shape[1]} columns.")

print("\n")
print(df_clean.info())


Dataset loaded successfully with 48895 rows and 16 columns.


<class 'pandas.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  str    
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  str    
 4   neighbourhood_group             48895 non-null  str    
 5   neighbourhood                   48895 non-null  str    
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  str    
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-

### Data Type Error
In this test, we verify that an attribute contains the correct underlying data type. In our Airbnb dataset, the `minimum_nights` column represents a count of days and must strictly be a numerical type (integer). We will simulate data type errors by injecting string values into this column.

In [11]:
# Introduce error in ~5% of the rows for the 'minimum_nights' column
np.random.seed(42) 
error_ratio = 0.05
total_rows = len(df_dirty)
num_errors = int(total_rows * error_ratio)

# Randomly select indices to corrupt
error_indices = np.random.choice(df_dirty.index, size=num_errors, replace=False)

# Explicitly cast the column to 'object' so Pandas allows mixed data types
df_dirty['minimum_nights'] = df_dirty['minimum_nights'].astype(object)

# Inject a string value where a number should be
df_dirty.loc[error_indices, 'minimum_nights'] = 'invalid_string'

print(f"Injected data type errors into {num_errors} rows in the 'minimum_nights' column.")

Injected data type errors into 2444 rows in the 'minimum_nights' column.


In [20]:
# Checker code to find data type errors
# Attempting to convert the column to numeric. Strings that fail conversion become NaN.
numeric_conversion = pd.to_numeric(df_dirty['minimum_nights'], errors='coerce')
invalid_type_mask = numeric_conversion.isna()

# Extract the rows where the error was detected
detected_type_errors = df_dirty[invalid_type_mask]

In [21]:
# Report of findings
print(f"The validity checker successfully detected {len(detected_type_errors)} data type errors.")
print("The automated check identified rows containing non-numeric values ('invalid_string') where an integer for minimum nights was expected.")
print("\nExamples of the detected invalid data points:")

# Displaying 3 examples of the corrupted data
display(detected_type_errors[['name', 'minimum_nights']].head(3))

The validity checker successfully detected 2444 data type errors.
The automated check identified rows containing non-numeric values ('invalid_string') where an integer for minimum nights was expected.

Examples of the detected invalid data points:


,name,minimum_nights
4,Entire Apt: Spacious Studio/Loft by central park,invalid_string
62,2 bedroom - Upper East Side-great for kids,invalid_string
144,FLAT MACDONOUGH GARDEN,invalid_string


### Range Error
In this test, we verify that numerical values fall within a logically acceptable minimum and maximum range. The `price` column represents the nightly cost of the listing. Since a price cannot be negative, the valid range must be `>= 0`. We will intentionally inject negative values to simulate range errors.

In [13]:
# Introduce range errors in ~5% of the rows for the 'price' column by injecting negative values
error_ratio = 0.05
total_rows = len(df_dirty)
num_errors = int(total_rows * error_ratio)

error_indices = np.random.choice(df_dirty.index, size=num_errors, replace=False)

# Inject negative values by multiplying the original price by -1 and subtracting 50
df_dirty.loc[error_indices, 'price'] = (df_dirty.loc[error_indices, 'price'] * -1) - 50

print(f"Injected negative prices (range errors) into {num_errors} rows in the 'price' column.")

Injected negative prices (range errors) into 2444 rows in the 'price' column.


In [22]:
# Checker code to find range errors
# Filter the dataframe to find any rows where 'price' is less than 0
range_error_mask = df_dirty['price'] < 0

# Extract the rows where the error was detected
detected_range_errors = df_dirty[range_error_mask]

In [23]:
# Report of findings 
print(f"The validity checker successfully detected {len(detected_range_errors)} range errors.")
print("The automated check identified rows where the nightly price was logically impossible (less than zero).")
print("\nExamples of the detected invalid data points:")

display(detected_range_errors[['name', 'price']].head(3))

The validity checker successfully detected 2444 range errors.
The automated check identified rows where the nightly price was logically impossible (less than zero).

Examples of the detected invalid data points:


,name,price
15,Only 2 stops to Manhattan studio,-190
64,Double Room w Private Deck Clinton Hill Best Area,-105
70,SpaHa Loft: Enormous and Bright,-275


### Format Error
In this test, we verify that string values adhere to a specific expected pattern or format. The `last_review` column contains dates that should be formatted strictly as `YYYY-MM-DD`. We will introduce formatting errors by injecting completely malformed date strings into this column.

In [17]:
# Introduce format errors in ~5% of the rows for the 'last_review' column by injecting a string that does not match the expected date format
np.random.seed(44)
error_ratio = 0.05
total_rows = len(df_dirty)
num_errors = int(total_rows * error_ratio)

error_indices = np.random.choice(df_dirty.index, size=num_errors, replace=False)

# Inject a format-breaking string
df_dirty.loc[error_indices, 'last_review'] = 'DD-MM-YYYY-WRONG'

print(f"Injected format errors into {num_errors} rows in the 'last_review' column.")

Injected format errors into 2444 rows in the 'last_review' column.


In [24]:
# Checker code to find format errors
# We use regex to ensure the string matches the YYYY-MM-DD pattern.
# We must first dropna() to avoid flagging naturally missing reviews as format errors.
non_null_reviews = df_dirty['last_review'].dropna()

# Check against the expected YYYY-MM-DD format
valid_format_mask = non_null_reviews.str.match(r'^\d{4}-\d{2}-\d{2}$')

# Find the indices that do NOT match the format
invalid_format_indices = valid_format_mask[~valid_format_mask].index

detected_format_errors = df_dirty.loc[invalid_format_indices]

In [25]:
# Report of findings 
print(f"The validity checker successfully detected {len(detected_format_errors)} format errors.")
print("The automated regex check caught date strings that violated the YYYY-MM-DD structure.")
print("\nExamples of the detected invalid data points:")

display(detected_format_errors[['name', 'last_review']].head(3))

The validity checker successfully detected 2444 format errors.
The automated regex check caught date strings that violated the YYYY-MM-DD structure.

Examples of the detected invalid data points:


,name,last_review
0,Clean & quiet apt home by the park,DD-MM-YYYY-WRONG
14,West Village Nest - Superhost,DD-MM-YYYY-WRONG
48,bright and stylish duplex,DD-MM-YYYY-WRONG


### Consistency Error
In this test, we verify that the logical relationship between two or more attributes holds true. In our dataset, if a listing has a `number_of_reviews` equal to 0, its `reviews_per_month` should logically be 0 or NaN. We will introduce consistency errors by forcing the total reviews to 0 while leaving a positive monthly review rate intact.

In [26]:
# Introduce consistency errors in ~5% of the rows
np.random.seed(45) 
error_ratio = 0.05
total_rows = len(df_dirty)
num_errors = int(total_rows * error_ratio)

# To make sure we create a contradiction, we only target rows that currently have > 0 reviews
valid_review_indices = df_dirty[df_dirty['number_of_reviews'] > 0].index

# Randomly select indices from the valid ones
error_indices = np.random.choice(valid_review_indices, size=num_errors, replace=False)

# Inject the error: Force total reviews to 0, creating a contradiction with reviews_per_month
df_dirty.loc[error_indices, 'number_of_reviews'] = 0

print(f"Injected consistency errors into {num_errors} rows by contradicting review counts.")

Injected consistency errors into 2444 rows by contradicting review counts.


In [27]:
# Checker code to find consistency errors
# We look for rows where number_of_reviews is 0, BUT reviews_per_month is greater than 0
consistency_error_mask = (df_dirty['number_of_reviews'] == 0) & (df_dirty['reviews_per_month'] > 0)

# Extract the rows where the error was detected
detected_consistency_errors = df_dirty[consistency_error_mask]

In [28]:
# Report of findings
print(f"The validity checker successfully detected {len(detected_consistency_errors)} consistency errors.")
print("The automated check identified rows where the mathematical relationship between total reviews and monthly reviews was logically broken.")
print("\nExamples of the detected invalid data points (Notice 0 total reviews but > 0 monthly reviews):")

display(detected_consistency_errors[['name', 'number_of_reviews', 'reviews_per_month']].head(3))

The validity checker successfully detected 2444 consistency errors.
The automated check identified rows where the mathematical relationship between total reviews and monthly reviews was logically broken.

Examples of the detected invalid data points (Notice 0 total reviews but > 0 monthly reviews):


,name,number_of_reviews,reviews_per_month
6,BlissArtsSpace!,0,0.40
51,Cozy 1BD on Central Park West in New York City,0,0.63
66,Light-filled 2B duplex in the heart of Park Sl...,0,0.16


### Uniqueness Error
In this test, we verify that an attribute which acts as a primary identifier contains no duplicates. The `id` column in our dataset is the unique identifier for each Airbnb listing. We will introduce uniqueness errors by overwriting the IDs of multiple rows with a single duplicated ID value.

In [32]:
# Introduce uniqueness errors in ~5% of the rows for the 'id' column
np.random.seed(46) 
error_ratio = 0.05
total_rows = len(df_dirty)
num_errors = int(total_rows * error_ratio)

# Randomly select indices to corrupt
error_indices = np.random.choice(df_dirty.index, size=num_errors, replace=False)

# Grab a single valid ID from the first row of the dataset to act as our duplicate
duplicate_id = df_dirty['id'].iloc[0]

# Overwrite the IDs of the randomly selected rows with this duplicate ID
df_dirty.loc[error_indices, 'id'] = duplicate_id

print(f"Injected uniqueness errors into {num_errors} rows using duplicated ID: {duplicate_id}.")

Injected uniqueness errors into 2444 rows using duplicated ID: 2539.


In [33]:
# Checker code to find uniqueness errors
# We use the duplicated() method. keep=False ensures we flag ALL instances of the duplicate, 
# not just the subsequent ones.
duplicate_mask = df_dirty.duplicated(subset=['id'], keep=False)

# Extract the rows where the error was detected
detected_uniqueness_errors = df_dirty[duplicate_mask]

In [34]:
# Report of findings
print(f"The validity checker successfully detected {len(detected_uniqueness_errors)} uniqueness errors.")
print("The automated check found multiple distinct rows sharing the exact same primary key (listing ID).")
print("\nExamples of the detected invalid data points (Notice the identical IDs):")

display(detected_uniqueness_errors[['id', 'name', 'host_id']].head(3))

The validity checker successfully detected 2445 uniqueness errors.
The automated check found multiple distinct rows sharing the exact same primary key (listing ID).

Examples of the detected invalid data points (Notice the identical IDs):


,id,name,host_id
0,2539,Clean & quiet apt home by the park,2787
4,2539,Entire Apt: Spacious Studio/Loft by central park,7192
20,2539,Sweet and Spacious Brooklyn Loft,21207


### Presence Error
In this test, we verify that mandatory attributes are actually present and not missing. For an Airbnb listing, the `host_name` is a required field. We will introduce presence errors by intentionally deleting the host names (setting them to null/NaN) in a portion of the dataset.

In [35]:
# Introduce presence errors in ~5% of the rows for the 'host_name' column
np.random.seed(47) 
error_ratio = 0.05
total_rows = len(df_dirty)
num_errors = int(total_rows * error_ratio)

error_indices = np.random.choice(df_dirty.index, size=num_errors, replace=False)

# Inject the error: replace the existing name with NaN (Not a Number/Null)
df_dirty.loc[error_indices, 'host_name'] = np.nan

print(f"Injected presence errors (missing data) into {num_errors} rows in the 'host_name' column.")

Injected presence errors (missing data) into 2444 rows in the 'host_name' column.


In [36]:
# Checker code to find presence errors
# We use the isna() method to flag any rows where the host_name is missing.
presence_error_mask = df_dirty['host_name'].isna()

# Extract the rows where the error was detected
detected_presence_errors = df_dirty[presence_error_mask]

In [45]:
# Report of findings
print(f"The validity checker successfully detected {len(detected_presence_errors)} presence errors.")
print("Note: The number of detected errors is slightly higher than the injected amount because the original dataset inherently contained a few missing host names.")
print("The automated check identified rows where the mandatory 'host_name' field was missing/null.")
print("\nExamples of the detected invalid data points (Notice the NaN in host_name):")

display(detected_presence_errors[['id', 'name', 'host_name']].head(3))

The validity checker successfully detected 2463 presence errors.
Note: The number of detected errors is slightly higher than the injected amount because the original dataset inherently contained a few missing host names.
The automated check identified rows where the mandatory 'host_name' field was missing/null.

Examples of the detected invalid data points (Notice the NaN in host_name):


,id,name,host_name
11,5441,Central Manhattan/near Broadway,NaN
21,8024,CBG CtyBGd HelpsHaiti rm#1:1-4,NaN
43,12318,West Side Retreat,NaN


### Length Error
In this test, we verify that string values do not exceed a logical maximum character length. The `name` column represents the title of the listing, which should realistically fit within standard database limits (e.g., 255 characters). We will introduce length errors by injecting artificially massive strings into this column.

In [38]:
# Introduce length errors in ~5% of the rows for the 'name' column
np.random.seed(48) 
error_ratio = 0.05
total_rows = len(df_dirty)
num_errors = int(total_rows * error_ratio)

error_indices = np.random.choice(df_dirty.index, size=num_errors, replace=False)

# Inject the error: A string of 500 "A"s
massive_string = "A" * 500
df_dirty.loc[error_indices, 'name'] = massive_string

print(f"Injected length errors into {num_errors} rows in the 'name' column.")

Injected length errors into 2444 rows in the 'name' column.


In [39]:
# Checker code to find length errors
# We check the length of the string and flag any that exceed 255 characters.
# We fill NaNs with 0 temporarily so the len() function doesn't fail on missing names.
length_error_mask = df_dirty['name'].str.len().fillna(0) > 255

# Extract the rows where the error was detected
detected_length_errors = df_dirty[length_error_mask]

In [40]:
# Report of findings
print(f"The validity checker successfully detected {len(detected_length_errors)} length errors.")
print("The automated check identified listing titles that exceeded the maximum allowed length of 255 characters.")
print("\nExamples of the detected invalid data points (Strings are truncated for display):")

# Displaying the first 50 characters of the name to keep the notebook clean
pd.set_option('display.max_colwidth', 50)
display(detected_length_errors[['id', 'name', 'host_id']].head(3))
pd.reset_option('display.max_colwidth') # Resetting to default

The validity checker successfully detected 2444 length errors.
The automated check identified listing titles that exceeded the maximum allowed length of 255 characters.

Examples of the detected invalid data points (Strings are truncated for display):


,id,name,host_id
7,5178,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,8967
38,11943,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,45445
83,19282,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,73469


### Look-up Error
In this test, we verify that categorical values strictly belong to a predefined list of acceptable options. The `room_type` column should only contain "Private room", "Entire home/apt", or "Shared room". We will introduce look-up errors by injecting a completely fabricated category that doesn't exist in the accepted list.

In [46]:
# Introduce look-up errors in ~5% of the rows for the 'room_type' column
np.random.seed(49) 
error_ratio = 0.05
total_rows = len(df_dirty)
num_errors = int(total_rows * error_ratio)

error_indices = np.random.choice(df_dirty.index, size=num_errors, replace=False)

# Inject the error: A fake category
df_dirty.loc[error_indices, 'room_type'] = 'Cardboard box'

print(f"Injected look-up errors into {num_errors} rows in the 'room_type' column.")

Injected look-up errors into 2444 rows in the 'room_type' column.


In [47]:
# Checker code to find look-up errors
# Define the accepted dictionary/look-up list
valid_room_types = ['Private room', 'Entire home/apt', 'Shared room']

# Flag rows where the room_type is NOT IN the valid list
lookup_error_mask = ~df_dirty['room_type'].isin(valid_room_types)

# Extract the rows where the error was detected
detected_lookup_errors = df_dirty[lookup_error_mask]

In [49]:
# Report of findings
print(f"The validity checker successfully detected {len(detected_lookup_errors)} look-up errors.")
print("The automated check caught categorical values ('Cardboard box') that do not belong to the accepted room types.")
print("\nExamples of the detected invalid data points:")

display(detected_lookup_errors[['id', 'name', 'room_type']].head(3))

The validity checker successfully detected 2444 look-up errors.
The automated check caught categorical values ('Cardboard box') that do not belong to the accepted room types.

Examples of the detected invalid data points:


,id,name,room_type
10,5295,Beautiful 1br on Upper West Side,Cardboard box
31,9704,Spacious 1 bedroom in luxe building,Cardboard box
39,12048,LowerEastSide apt share shortterm 1,Cardboard box


### Dataset 2
Description: Starbucks Customer Ordering transactions from a two year period between 2024 and 2025

Dataset Name: Starbucks Customer Ordering Patterns Dataset


Author: Compiled by Kaggle user Likitha Gedipudi, utilizing public data originally scraped from Airbnb.


Purpose: This is a real-world, non-synthetic dataset created to explore, visualize, and analyze Airbnb listing activity, pricing metrics, and geographic trends across New York City.


Shape: The dataset contains exactly 48,895 rows and 16 columns.

### References

New York City Airbnb open data. (2019). Kaggle. https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data 

pandas.DataFrame.duplicated — pandas 3.0.1 documentation. (n.d.). https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html 

pandas.to_numeric — pandas 3.0.1 documentation. (n.d.). https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html

Google Gemini, queries preformed: 

"How to randomly select exactly 5% of row indices in a Pandas dataframe without replacement using numpy?" (Used to generate the random indexing logic for noise introduction).

"Pandas how to fix 'TypeError: Invalid value 'invalid_string' for dtype 'int64'' when trying to inject a string into a numeric column?" (Used to find the .astype(object) workaround for the Data Type error test).

"How to find all duplicate rows based on a specific ID column in Pandas, making sure to include the original first occurrence in the mask?" (Used to discover the keep=False parameter for the Uniqueness error test).